# 1. 数据提供管道

- .h5 是一种数据文件格式，全名是 HDF5（Hierarchical Data Format version 5）。它不是图片文件，也不是普通文本文件，而是一种专门用于存储大规模科学数据、机器学习数据和实验数据的容器格式。
- STEMNIST中的.h5结构：
```
AT_A_1.h5

└── pressure_data
        |
        └── numpy array
              shape=(240,16,16)
              dtype=uint8
```

In [ ]:
from collections import Counter
from hashlib import md5 # 用于检查原始ZIP文件是否完好
from pathlib import Path
from zipfile import ZipFile

import h5py # 用于读取.h5文件
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset

In [12]:
# 数据路径
MYCODE_DIR = Path.cwd().resolve()
# print("Notebook 当前工作目录：", MYCODE_DIR)
# SNN_for_STEMNIST/first_stage
FIRST_STAGE_DIR = MYCODE_DIR.parent

# SNN_for_STEMNIST/first_stage/STEMNIST
DATA_ROOT = FIRST_STAGE_DIR / "STEMNIST"
ZIP_PATH = DATA_ROOT / "STEMNIST Dataset.zip"
EXTRACT_ROOT = DATA_ROOT / "extracted"
RAW_DIR = EXTRACT_ROOT / "STEMNIST Dataset" / "RawCharacters"

# 预期的ZIP文件MD5值，用于验证文件完整性
EXPECTED_MD5 = "6ca4638b2f95bf34f59873ab62399bd8"   
EXPECTED_SHAPE = (240, 16, 16)

In [13]:
# 标签顺序一旦确定，之后不能改变
LABELS = tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ123456789")
LABEL_TO_INDEX = {
    label: index
    for index, label in enumerate(LABELS)
}

In [14]:
# 根据Zip数据文件计算其MD5值，用于验证文件完整性
def calculate_md5(path):
    """分块计算 ZIP 文件的 MD5。"""

    digest = md5()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

## 解压STEMNIST Dataset.zip并返回成records列表

In [15]:
def prepare_and_scan_samples():
    """校验、解压并扫描所有原始压力样本。"""

    if not ZIP_PATH.is_file():
        raise FileNotFoundError(
            f"找不到数据压缩包：{ZIP_PATH.resolve()}"
        )

    actual_md5 = calculate_md5(ZIP_PATH)

    if actual_md5 != EXPECTED_MD5:
        raise RuntimeError(
            f"ZIP 文件 MD5 校验失败：\n"
            f"期望：{EXPECTED_MD5}\n"
            f"实际：{actual_md5}"
        )

    # 只有尚未解压时才执行解压
    if not RAW_DIR.is_dir():
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

        with ZipFile(ZIP_PATH, "r") as archive:
            archive.extractall(EXTRACT_ROOT)

    paths = sorted(RAW_DIR.glob("*.h5"))

    if len(paths) != 7700:
        raise RuntimeError(
            f"应有 7700 个原始样本，实际找到 {len(paths)} 个"
        )

    records = []

    for path in paths:
        # 例如：AT_A_1.h5
        parts = path.stem.split("_")

        if len(parts) != 3:
            raise RuntimeError(f"无法解析文件名：{path.name}")

        participant_id, label, repetition = parts

        if label not in LABEL_TO_INDEX:
            raise RuntimeError(
                f"{path.name} 中出现未知标签：{label}"
            )

        # 先检查 HDF5 的内部结构
        with h5py.File(path, "r") as file:
            if "pressure_data" not in file:
                raise RuntimeError(
                    f"{path.name} 缺少 pressure_data"
                )

            pressure_dataset = file["pressure_data"]

            if pressure_dataset.shape != EXPECTED_SHAPE:
                raise RuntimeError(
                    f"{path.name} 形状错误："
                    f"{pressure_dataset.shape}"
                )

            if pressure_dataset.dtype != np.uint8:
                raise RuntimeError(
                    f"{path.name} 类型错误："
                    f"{pressure_dataset.dtype}"
                )

        records.append(
            {
                "sample_id": path.stem, # AT_A_1
                "participant_id": participant_id,
                "label": label,
                "label_index": LABEL_TO_INDEX[label],
                "repetition": int(repetition),
                "path": path,
            }
        )

    class_counts = Counter(
        record["label"] for record in records
    )
    participant_ids = {
        record["participant_id"] for record in records
    }

    if set(class_counts) != set(LABELS):
        raise RuntimeError("数据集没有完整包含固定的 35 类")

    if any(count != 220 for count in class_counts.values()):
        raise RuntimeError(
            f"每类应有 220 个样本，实际为：{dict(class_counts)}"
        )

    if len(participant_ids) != 34:
        raise RuntimeError(
            f"应有 34 名参与者，实际找到 {len(participant_ids)} 名"
        )

    return records

In [16]:
# test
records = prepare_and_scan_samples()

In [18]:
import pandas as pd
df = pd.DataFrame(records)
df.head()

,sample_id,participant_id,label,label_index,repetition,path
0,AT_1_1,AT,1,26,1,C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST\fi...
1,AT_1_10,AT,1,26,10,C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST\fi...
2,AT_1_2,AT,1,26,2,C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST\fi...
3,AT_1_3,AT,1,26,3,C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST\fi...
4,AT_1_4,AT,1,26,4,C:\Users\Fortyfour\Desktop\SNN_for_STEMNIST\fi...


## Dataset

In [19]:
class STEMNISTDataset(Dataset):
    """STEMNIST 原始连续压力 Dataset。"""

    def __init__(
        self,
        records,
        time_steps=240,
    ):
        if time_steps != 240:
            raise ValueError(
                "time_steps 目前只支持 240"
            )

        self.records = records
        self.time_steps = time_steps

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        with h5py.File(record["path"], "r") as file:
            pressure = file["pressure_data"][...]
            pressure = pressure.astype(np.float32)


        # 增加单通道维度：
        # [T,H,W] -> [T,C,H,W]
        x = torch.from_numpy(pressure).unsqueeze(1)

        y = torch.tensor(
            record["label_index"],
            dtype=torch.long,
        )

        # sample_id 不进入模型，但之后保存划分和预测时会用到
        return x, y, record["sample_id"]

## DataLoader

In [21]:
def make_dataloaders(
    time_steps=240,
    batch_size=64,
):
    records = prepare_and_scan_samples()

    labels = np.array(
        [record["label_index"] for record in records]
    )
    all_indices = np.arange(len(records))

    # 第一次划分：80% train_val，20% test
    train_val_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=42,
        stratify=labels,    # stratify 参数会根据你提供的类别标签进行分层，使划分后的子集尽量保持原始类别比例。
    )

    # 第二次划分：从 train_val 中取 20% 作为 val
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.20,
        random_state=42,
        stratify=labels[train_val_indices],
    )

    dataset = STEMNISTDataset(
        records,
        time_steps=time_steps,
    )

    # 固定训练集打乱的随机数生成器
    generator = torch.Generator().manual_seed(42)

    common_arguments = {
        "batch_size": batch_size,
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
    }

    train_loader = DataLoader(
        Subset(dataset, train_indices.tolist()),
        shuffle=True,
        generator=generator,
        **common_arguments,
    )

    val_loader = DataLoader(
        Subset(dataset, val_indices.tolist()),
        shuffle=False,
        **common_arguments,
    )

    test_loader = DataLoader(
        Subset(dataset, test_indices.tolist()),
        shuffle=False,
        **common_arguments,
    )

    return train_loader, val_loader, test_loader

In [22]:
# test
train_loader, val_loader, test_loader = make_dataloaders()

In [25]:
for x, y, sample_id in train_loader:
    print(x.shape, y.shape, sample_id)
    break

torch.Size([64, 240, 1, 16, 16]) torch.Size([64]) ['YX_S_3', 'SD_A_8', 'AT_5_8', 'SH_M_8', 'LM_E_3', 'ZF_Q_1', 'CZ_S_1', 'DA_C_3', 'KY_K_1', 'YX_4_1', 'ZX_P_1', 'HM_5_3', 'XL_Z_4', 'LG_A_4', 'ZX_5_2', 'JY_A_5', 'LG_D_5', 'RH_W_2', 'ML_2_2', 'TT_I_2', 'WG_F_5', 'YX_I_2', 'PZ_7_4', 'LG_R_10', 'JY_G_6', 'ML_B_4', 'JZ_S_2', 'XW_F_3', 'YY_N_1', 'SD_S_9', 'HM_L_5', 'XW_W_5', 'MY_B_5', 'ML_T_3', 'WT_R_5', 'YJ_C_4', 'ZF_F_10', 'XL_Y_3', 'SD_X_9', 'DA_T_2', 'ZZ_D_1', 'LY_D_4', 'SH_V_7', 'YJ_A_5', 'KY_Q_4', 'ML_I_3', 'TT_B_4', 'PZ_4_9', 'WL_Z_5', 'ZX_1_4', 'XL_G_5', 'PZ_5_4', 'AT_5_10', 'LG_6_6', 'LK_C_5', 'XW_B_2', 'ML_Q_3', 'QM_U_5', 'ZF_D_9', 'KY_V_9', 'BZ_C_1', 'LG_8_3', 'AT_N_3', 'AT_U_6']
